# JW $\mathbb{Z}_2^F \times \mathbb{Z}_2^T$

Created: 08-10-2026

Objectives:
* Iterate on [this notebook](jw_z_2_f_x_z_2_t_system.ipynb), but now apply random unitaries to the states.

# Imports

In [1]:
import numpy as np

In [2]:
import jax
jax.config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from operator import mul
from itertools import product

In [6]:
from random import random

In [7]:
import quimb.tensor as qtn
import quimb as qu

In [8]:
from scipy.stats import unitary_group

In [9]:
from collections import Counter

In [10]:
import pandas as pd

In [11]:
from time import time

In [12]:
from humanize import naturalsize

# Definitions
## Construct states

### Z product state

In [13]:
np_up_Z_state = np.array([1,0])

In [14]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='prod'
)

In [15]:
def get_product_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_Z_state.reindex({'k': f'k{i}'})
        for i in range(num_sites)
    ]

    out = qtn.TensorNetwork(
        product_state_tensors,
        virtual=True
    )
    out.mangle_inner_()

    return out

### XZX cluster state

In [16]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [17]:
qu_up_X_state = qtn.Tensor(
    data=np_up_X_state,
    inds=('k',),
    tags='prod'
)

In [18]:
np_CZ = np.diag([1,1,1,-1])

In [19]:
np_CZ = np_CZ.reshape((2,)*4)

In [20]:
qu_CZ = qtn.Tensor(
    data=np_CZ,
    inds=('k1', 'k2', 'b1', 'b2'),
    tags='CZ'
)

In [21]:
np_hadamard = np.pow(2, -1/2)*np.array([
    [1,1],
    [1,-1]
])

In [22]:
qu_hadamard = qtn.Tensor(data=np_hadamard, inds=('k', 'b'), tags='Had')

In [23]:
def get_xzx_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'kc_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_1_{i}',
            'b2': f'kc_1_{i+1}',
            'k1': f'kc_2_{i}',
            'k2': f'kc_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_2_{i}',
            'b2': f'kc_2_{(i+1)%num_sites}',
            'k1': f'k_3_{i}',
            'k2': f'k_3_{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'k_3_{i}',
            'k': f'k{i}'
        })
        for i in range(num_sites)
    ]
    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + hadamard_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

### YZY cluster state

In [24]:
np_S = np.array([
    [1,0],
    [0,1j]
])

In [25]:
np_xyz_rotator = np_S @ np_hadamard

In [26]:
qu_xyz_rotator = qtn.Tensor(data=np_xyz_rotator, inds=('k', 'b'), tags='xyz_rot')

In [27]:
def get_yzy_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'kc_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_1_{i}',
            'b2': f'kc_1_{i+1}',
            'k1': f'kc_2_{i}',
            'k2': f'kc_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_2_{i}',
            'b2': f'kc_2_{(i+1)%num_sites}',
            'k1': f'k_3_{i}',
            'k2': f'k_3_{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    rotator_layer = [
        qu_xyz_rotator.reindex({
            'b': f'k_3_{i}',
            'k': f'k{i}'
        })
        for i in range(num_sites)
    ]
    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + rotator_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

### XZZZX cluster state
$$ \ket\psi = H^{\otimes N} S^{\otimes N} (\prod_j CZ_{j,j+1}) H^{\otimes N} S^{\dagger \otimes N} (\prod_j CZ_{j,j+1}) \ket{+}^{\otimes N}$$

In [28]:
qu_S = qtn.Tensor(data=np_S, inds=('k', 'b'), tags='S')

In [29]:
def get_xzzzx_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'k_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'k_1_{i}',
            'b2': f'k_1_{i+1}',
            'k1': f'k_2_{i}',
            'k2': f'k_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'k_2_{i}',
            'b2': f'k_2_{(i+1)%num_sites}',
            'k1': f'k_3_{i}',
            'k2': f'k_3_{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    first_s_layer = [
        qu_S.conj().reindex({
            'b': f'k_3_{i}',
            'k': f'k_4_{i}'
        })
        for i in range(num_sites)
    ]

    first_hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'k_4_{i}',
            'k': f'k_5_{i}'
        })
        for i in range(num_sites)
    ]

    third_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'k_5_{i}',
            'b2': f'k_5_{i+1}',
            'k1': f'k_6_{i}',
            'k2': f'k_6_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]

    fourth_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'k_6_{i}',
            'b2': f'k_6_{(i+1)%num_sites}',
            'k1': f'k_7_{i}',
            'k2': f'k_7_{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    second_s_layer = [
        qu_S.reindex({
            'b': f'k_7_{i}',
            'k': f'k_8_{i}'
        })
        for i in range(num_sites)
    ]

    second_hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'k_8_{i}',
            'k': f'k{i}'
        })
        for i in range(num_sites)
    ]

    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + first_s_layer
        + first_hadamard_layer
        + third_layer_circuit_tensors
        + fourth_layer_circuit_tensors
        + second_s_layer
        + second_hadamard_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

## Apply random unitary to groundstate

In [30]:
def generate_random_o2():
    rng = np.random.default_rng()

    theta = rng.uniform(0, 2*np.pi)
    s = rng.choice([-1, 1])

    c, sn = np.cos(theta), np.sin(theta)

    return np.array([
        [c, -sn],
        [s*sn, s*c],
    ])


def generate_random_symmetry_respecting_unitary():
    O_even = generate_random_o2()
    O_odd  = generate_random_o2()

    U = np.zeros((4, 4))

    # even sector: |00>, |11>
    U[np.ix_([0, 3], [0, 3])] = O_even

    # odd sector: |01>, |10>
    U[np.ix_([1, 2], [1, 2])] = O_odd

    return U

In [31]:
X = generate_random_symmetry_respecting_unitary()

In [32]:
X

array([[ 0.19216889,  0.        ,  0.        , -0.98136187],
       [ 0.        ,  0.97153771, -0.23688495,  0.        ],
       [ 0.        , -0.23688495, -0.97153771,  0.        ],
       [-0.98136187,  0.        ,  0.        , -0.19216889]])

In [33]:
np.round(X @ (X.conj().T), 3)

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

Now reshape tensor.

In [34]:
def generate_random_symmetry_respecting_qu_unitary():
    # Generate a unitary which commutes with ZZ, and K

    X = generate_random_symmetry_respecting_unitary()
    X = X.reshape((2,2,2,2))

    qu_u = qtn.Tensor(
        X,
        inds=['k1', 'k2', 'b1', 'b2']
    )
    
    return qu_u

In [35]:
# Warning, likely making assupmtions about shape of psi, number of sites being even here etc.
def apply_haar_random_fdlu_to_quimb_state(psi, domains_dict):
    num_sites = domains_dict['num_system_sites']

    depth = domains_dict['fdlu_depth']
    offset = domains_dict['fdlu_offset']
    all_circuit_lists = [
        list() for _ in range(depth)
    ]

    for layer, circuit_list in enumerate(all_circuit_lists):
        delta = layer

        for i in range(num_sites//2):
            site_1 = ((2*i)+delta+offset)%num_sites
            site_2 = ((2*i)+1+delta+offset)%num_sites

            u = generate_random_symmetry_respecting_qu_unitary()

            reindex_map = {
                'k1': f'k_{layer+1}_{site_1}',
                'k2': f'k_{layer+1}_{site_2}',
                'b1': f'k_{layer}_{site_1}',
                'b2': f'k_{layer}_{site_2}'
            }
            
            qu_u = u.reindex(reindex_map)

            circuit_list.append(qu_u)

    all_tensors = (
        [psi.reindex({f'k{i}': f'k_0_{i}' for i in range(num_sites)})]
        + sum(all_circuit_lists, start=[])
    )

    out = (
        qtn
        .TensorNetwork(all_tensors, virtual=False)
        .mangle_inner_()
        .reindex({f'k_{depth}_{i}': f'k{i}' for i in range(num_sites)}) 
    )

    return out

## Symmetries

In [36]:
np_I = np.array([
    [1,0],
    [0,1]
])

np_X = np.array([
    [0,1],
    [1,0]
])

np_Y = np.array([
    [0,-1j],
    [1j,0]
])

np_Z = np.array([
    [1,0],
    [0,-1]
])

In [37]:
qu_I = qtn.Tensor(
    np_I,
    inds=['k', 'b'],
    tags='X'
)

qu_X = qtn.Tensor(
    np_X,
    inds=['k', 'b'],
    tags='X'
)

qu_Y = qtn.Tensor(
    np_Y,
    inds=['k', 'b'],
    tags='Y'
)

qu_Z = qtn.Tensor(
    np_Z,
    inds=['k', 'b'],
    tags='Z'
)

In [38]:
def multikron(arrays):
    return reduce(np.kron, arrays)

In [39]:
def get_multisite_qu_Z(num_sites):
    np_many_X = multikron([np_Z]*num_sites)

    out = qtn.Tensor(
        np_many_X,
        inds=['k', 'b'],
        tags='mulit_site_X',
    )

    return out

In [40]:
def get_multisite_qu_I(num_sites):
    np_many_I = multikron([np_I]*num_sites)

    out = qtn.Tensor(
        np_many_I,
        inds=['k', 'b'],
        tags='mulit_site_I',
    )

    return out

## Extracting projectors

In [41]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [42]:
def maximize_projector_states(rho, left_sites, proj_sites, right_sites):
    v = qtn.Tensor(
        data=random_uniform_complex((2,)*(len(proj_sites)-1)),
        inds=[f'k{i}' for i in proj_sites[:-1]]
    )

    tnopt = qtn.TNOptimizer(
        v,  # the tensor network we want to optimize
        loss_func,  # the function we want to minimize
        norm_fn=normalize_v,
        loss_constants={"rho": rho},
        loss_kwargs={
            "left_sites": left_sites,
            "proj_sites": proj_sites,
            "right_sites": right_sites,
        },
        autodiff_backend="jax",
        optimizer="L-BFGS-B",
        progbar=False
    )

    v_opt = tnopt.optimize(n=2000)

    embedded_v_opt = embed_fp_even_vector(v_opt).contract()

    return v_opt, embedded_v_opt, tnopt.losses

In [43]:
def projector_state_check(state, rho, left_sites, right_sites):
    left_right_rho = (
        rho
        & state
        & state.conj().reindex({s: f'b{s[1:]}' for s in state.inds})
    )

    # ChatGPT suggestion
    left_right_rho = left_right_rho.contract(
        output_inds=[
            f"{s}{i}"
            for s in "kb"
            for i in left_sites + right_sites
        ]
    )

    left_inds = [
        f'{s}{i}'
        for i in left_sites
        for s in 'kb'
    ]

    schmidt_decomp = qtn.tensor_core.tensor_split(
        left_right_rho,
        left_inds=left_inds,
        method='svd',
        #cutoff=1e-6,
        cutoff_mode='abs',
        absorb=None,
        renorm=False,
        bond_ind='v'
    )

    schmidt_vals = schmidt_decomp.tensors[1]

    return schmidt_vals

### Embed vector

In [44]:
np_00 = np.array([[1,0], [0,0]])
np_11 = np.array([[0,0], [0,1]])

In [45]:
def tensor_product_operators(op_1, op_2):
    out = (
        op_1[..., np.newaxis, np.newaxis]
        *op_2[np.newaxis, np.newaxis, ...]
    )

    return out

In [46]:
np_CX = (
    tensor_product_operators(np_00, np_I)
    + tensor_product_operators(np_11, np_X)
)

In [47]:
qu_CX = qtn.Tensor(
    np_CX,
    inds=['k1', 'b1', 'k2', 'b2']
)

In [48]:
np_up_Z_state = np.array([1,0])

In [49]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='Z0_pad'
)

In [50]:
def embed_fp_even_vector(v):
    # Take a vector of length 2N-1, and return a vector of length 2N
    # which commutes with ZZZ...ZZZ

    sites = sorted(int(s[1:]) for s in v.inds)
    padded_site = sites[-1] + 1
    num_cx_gates = len(sites)
    
    if num_cx_gates>=1:
        padded_v = (
            v
            & qu_up_Z_state.reindex({'k': f'k{padded_site}_0'})
        )
    else:
        padded_v = (
            v
            & qu_up_Z_state.reindex({'k': f'k{padded_site}'})
        )

    # Probably better with an enumerate
    cx_gates = [
        qu_CX.reindex({
            'k1': f'k{sites[i]}',
            'b1': f'b{sites[i]}',
            'k2': f'k{padded_site}_{i+1}',
            'b2': f'k{padded_site}_{i}'
        })
        for i in range(num_cx_gates-1)
    ]
    
    # Handling annoying edge case logic
    if num_cx_gates >= 1:
        i = num_cx_gates-1
        cx_gates.append(
            qu_CX.reindex({
                'k1': f'k{sites[i]}',
                'b1': f'b{sites[i]}',
                'k2': f'k{padded_site}',
                'b2': f'k{padded_site}_{i}'
            })
        )
    
    reindexed_padded_v = (
        padded_v
        .reindex({
            f'k{sites[i]}': f'b{sites[i]}'
            for i in range(num_cx_gates)
        })
    )
    sym_v = qtn.TensorNetwork([
        reindexed_padded_v,
        *cx_gates
    ])
    
    sym_v.mangle_inner_()

    return sym_v

### Loss function

In [51]:
def get_rho_purity(rho, sites):
    # Assuming rho is a Hermitian reduced density matrix
    reindex_map = (
        {f'k{i}': f'b{i}' for i in sites}
        | {f'b{i}': f'k{i}' for i in sites}
    )

    rho_other = rho.reindex(reindex_map)
    #rho_other.mangle_inner_()
    
    out = (rho & rho_other).contract()

    return out

In [52]:
def normalize_v(v):
    norm = (v & v.conj()).contract()
    w = v*jnp.power(norm, -0.5)
    return w

In [53]:
def loss_func(v, rho, left_sites, proj_sites, right_sites):
    embed_v = embed_fp_even_vector(v)
    
    rho_lr = (
        rho
        & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
        & embed_v.conj()
    )
    # ChatGPT suggestion
    rho_lr = rho_lr.contract(
        output_inds=[
            f"{s}{i}"
            for s in "kb"
            for i in left_sites + right_sites
        ]
    )
    tr_rho_lr = (
        rho_lr
        .reindex({f'k{i}': f'b{i}' for i in left_sites + right_sites})
        .contract()
    )
    
    rho_l = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in right_sites}
    )
    rho_l = rho_l.contract()*jnp.power(tr_rho_lr, -0.5)
    
    rho_r = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in left_sites}
    )
    rho_r = rho_r.contract()*jnp.power(tr_rho_lr, -0.5)
    
    purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
    purity_l = get_rho_purity(rho_l, left_sites)
    purity_r = get_rho_purity(rho_r, right_sites)
    
    reindex_map = (
        {f'k{i}': f'b{i}' for i in left_sites+right_sites}
        | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
    )
    
    cross_term = (
        rho_lr.reindex(reindex_map)
        & rho_l
        & rho_r
    )
    cross_term = cross_term.contract()

    out = jnp.real(
        (purity_lr + purity_l*purity_r - 2*cross_term)/
        purity_lr
    )

    return out

## Extract cut rho and EDM

Parametrize symmetries as $(p,g) \in \mathbb{Z}_2^F \times \mathbb{Z}_2^T$, with $p,g \in \mathbb{Z}_2$.

In [54]:
qu_Z

Tensor(shape=(2, 2), inds=('k', 'b'), tags=oset(['Z']))

In [55]:
qu_I

Tensor(shape=(2, 2), inds=('k', 'b'), tags=oset(['X']))

In [56]:
def generate_antilinear_edm_from_cut_state(cut_state, sites, num_defect_sites, fermion_parity=False):
    # Lots of duplicate code, probably a better way to do this.
    # Computes for U((p, 1)), where p is determined by fermion_parity.
    assert 2*num_defect_sites < len(sites)
    #assert (num_defect_sites%2) == 0
    #assert (len(sites)%2) == 0
    #assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    if fermion_parity:
        fp_unitary = qu_Z
    else:
        fp_unitary = qu_I

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        fp_unitary.reindex({
            'k': f'b{i}',
            'b': f'c{i}'
        })
        for i in left_defect_sites
    ]

    inner_gates = [
        fp_unitary.reindex({
            'k': f'k{i}',
            'b': f'b{i}',
        })
        for i in internal_sites
    ]

    right_sym_gates = [
        fp_unitary.reindex({
            'k': f'b{i}',
            'b': f'c{i}',
        })
        for i in right_defect_sites
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    # The fact that we don't conjugate the reindexed cut_state means that
    # we are effectively implementing local complex conjugation.
    edm = (
        cut_state
        & cut_state.reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

In [57]:
def generate_fp_edm_from_cut_state(cut_state, sites, num_defect_sites):
    # Lots of duplicate code, probably a better way to do this.
    # Computes for U((p, 1)), where p is determined by fermion_parity.
    assert 2*num_defect_sites < len(sites)
    #assert (num_defect_sites%2) == 0
    #assert (len(sites)%2) == 0
    #assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    fp_unitary = qu_Z

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        fp_unitary.reindex({
            'k': f'b{i}',
            'b': f'c{i}'
        })
        for i in left_defect_sites
    ]

    inner_gates = [
        fp_unitary.reindex({
            'k': f'k{i}',
            'b': f'b{i}',
        })
        for i in internal_sites
    ]

    right_sym_gates = [
        fp_unitary.reindex({
            'k': f'b{i}',
            'b': f'c{i}',
        })
        for i in right_defect_sites
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    edm = (
        cut_state
        & cut_state.conj().reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

In [58]:
def generate_all_edms_from_cut_state(cut_state, sites, num_defect_sites):
    args = (cut_state, sites, num_defect_sites)

    out = [
        generate_antilinear_edm_from_cut_state(*args, False),
        generate_fp_edm_from_cut_state(*args),
        generate_antilinear_edm_from_cut_state(*args, True)
    ]

    return out

## Defect operators

In [59]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [60]:
def solve_for_boundary_operators(edm, num_iters=100):
    # Careful, the indices are reversed here for ease.
    # i.e. the k, b indices have been swapped to make tensor contraction easier.
    scores = list()

    u_left = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_left'), edm.ind_size('k_left')),
        inds=['k_left', 'b_left'],
        dtype='complex64'
    )

    u_right = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_right'), edm.ind_size('k_right')),
        inds=['k_right', 'b_right'],
        dtype='complex64'
    )

    for _ in range(num_iters):
        right_edm = (edm & u_left).contract()
        data = right_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_right = qtn.Tensor(sol, inds = ['b_right', 'k_right'])

        left_edm = (edm & u_right).contract()
        data = left_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_left = qtn.Tensor(sol, inds = ['b_left', 'k_left'])

    return (u_left, u_right), scores

## Sweep function

In [61]:
def extract_projector(psi, rho_sites, num_pad_sites):
    proj_sites = rho_sites
    left_sites = list(range(
        min(rho_sites) - num_pad_sites,
        min(rho_sites)
    ))
    right_sites = list(range(
        max(rho_sites)+1,
        max(rho_sites)+num_pad_sites+1
    ))
    all_sites = left_sites + proj_sites + right_sites
    rho = (
        psi
        & psi.conj().reindex({f'k{i}': f'b{i}' for i in all_sites})
    )

    # ChatGPT suggestion for performance
    rho = rho.full_simplify(
        seq="ADCR",
        output_inds=[
            f"{s}{i}"
            for s in "kb"
            for i in all_sites
        ],
        atol=1e-12,
    )

    raw_proj_vec, proj_vec, losses = maximize_projector_states(
        rho,
        left_sites,
        proj_sites,
        right_sites
    )

    schmidt_vals = projector_state_check(
        proj_vec,
        rho,
        left_sites,
        right_sites
    )

    return raw_proj_vec, proj_vec, losses, schmidt_vals

In [62]:
def contract_purity_network(tn):
    tn = tn.full_simplify(
        seq="ADCR",
        output_inds=(),
        atol=1e-12,
    )
    return tn.contract(output_inds=())

In [63]:
def find_invariants_via_projectors_from_random_state(psi, domains_dict):
    rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
    
    raw_left_proj_vec, left_proj_vec, *left_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['left_projector_sites'],
        domains_dict['num_projector_pad_sites']
    )
    
    raw_right_proj_vec, right_proj_vec, *right_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['right_projector_sites'],
        domains_dict['num_projector_pad_sites']
    )
    
    cut_sites = list(range(
        min(domains_dict['left_projector_sites']),
        max(domains_dict['right_projector_sites'])+1
    ))
    
    cut_rho_unprojected = (
        rand_psi
        & rand_psi.conj().reindex({f'k{i}': f'b{i}' for i in cut_sites})
    )
    
    left_proj_sites = domains_dict['left_projector_sites']
    right_proj_sites = domains_dict['right_projector_sites']
    
    cut_rho = (
        cut_rho_unprojected.reindex({
            **{f'k{i}': f'l{i}' for i in left_proj_sites},
            **{f'k{i}': f'l{i}' for i in right_proj_sites},
            **{f'b{i}': f'c{i}' for i in left_proj_sites},
            **{f'b{i}': f'c{i}' for i in right_proj_sites}
        })
        & left_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in left_proj_sites})
        & left_proj_vec
        & left_proj_vec.reindex({f'k{i}': f'c{i}' for i in left_proj_sites})
        & left_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in left_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in right_proj_sites})
        & right_proj_vec
        & right_proj_vec.reindex({f'k{i}': f'c{i}' for i in right_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in right_proj_sites})
    )
    
    cut_rho_trace = (
        cut_rho
        .reindex({f'b{i}': f'k{i}' for i in cut_sites})
        .contract()
    )
    
    cut_rho = cut_rho/cut_rho_trace
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in cut_sites}
        | {f'b{i}': f'k{i}' for i in cut_sites}
    )
    
    cut_rho_purity = contract_purity_network(
        cut_rho & cut_rho.reindex(tranpose_map)
    )
    
    sub_cut_sites = list(range(
        max(domains_dict['left_projector_sites'])+1,
        min(domains_dict['right_projector_sites'])
    ))
    
    sub_cut_rho = (
        cut_rho
        .reindex({f'k{i}': f'b{i}' for i in left_proj_sites + right_proj_sites})
        #.contract()
    )
    
    sub_cut_rho_trace = (
        sub_cut_rho
        .reindex({f'b{i}': f'k{i}' for i in sub_cut_sites})
        .contract()
    )
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in sub_cut_sites}
        | {f'b{i}': f'k{i}' for i in sub_cut_sites}
    )
    
    sub_cut_rho_purity = contract_purity_network(
        sub_cut_rho & sub_cut_rho.reindex(tranpose_map)
    )
    
    cut_score, cut_state, cut_overlap = get_dominant_eigenvector(sub_cut_rho, False)

    cut_state_fermion_parity = compute_fermion_parity(cut_state, sub_cut_sites)

    edms = generate_all_edms_from_cut_state(
        cut_state,
        sub_cut_sites,
        domains_dict['num_defect_sites']
    )

    defect_ops_results = [
        solve_for_boundary_operators(
            edm,
            num_iters=20
        )
        for edm in edms
    ]

    left_defect_sites = list(range(
        max(domains_dict['left_projector_sites']) + 1,
        max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
    ))
    right_defect_sites = list(range(
        min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
        min(domains_dict['right_projector_sites'])
    ))
    
    left_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in left_defect_sites
        })
    )
    left_rdm = left_rdm.contract()
    left_fuse_map = [
        ('k_left', [f'k{i}' for i in left_defect_sites]),
        ('b_left', [f'b{i}' for i in left_defect_sites])
    ]
    left_rdm.fuse(left_fuse_map, inplace=True)
    
    right_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in right_defect_sites
        })
    )
    right_rdm = right_rdm.contract()
    right_fuse_map = [
        ('k_right', [f'k{i}' for i in right_defect_sites]),
        ('b_right', [f'b{i}' for i in right_defect_sites])
    ]
    right_rdm.fuse(right_fuse_map, inplace=True)

    defect_ops_pairs = [l[0] for l in defect_ops_results]
    left_defect_ops, right_defect_ops = list(zip(*defect_ops_pairs))

    np_left_rdm = (
        left_rdm
        .transpose('k_left', 'b_left')
        .data
    )

    np_left_defect_ops = [
        left_defect_op.transpose('k_left', 'b_left').data.T
        for left_defect_op in left_defect_ops
    ]

    np_right_rdm = (
        right_rdm
        .transpose('k_right', 'b_right')
        .data
    )
    
    np_right_defect_ops = [
        right_defect_op.transpose('k_right', 'b_right').data.T
        for right_defect_op in right_defect_ops
    ]

    np_left_defect_ops = [
        [
            np.identity(np_left_defect_ops[0].shape[0]),
            np_left_defect_ops[0]
        ],
        [
            np_left_defect_ops[1],
            np_left_defect_ops[2]
        ]
    ]
    np_right_defect_ops = [
        [
            np.identity(np_right_defect_ops[0].shape[0]),
            np_right_defect_ops[0]
        ],
        [
            np_right_defect_ops[1],
            np_right_defect_ops[2]
        ]
    ]

    """
    #Ignore for now
    if domains_dict['normalize_defect_op']:
        np_left_defect_op = normalize_defect_op(
            np_left_defect_op,
            np_left_rdm
        )
        right_defect_op = normalize_defect_op(
            np_right_defect_op,
            np_right_rdm
        )
    """

    np_fp = get_multisite_qu_Z(
        domains_dict['num_defect_sites']
    ).transpose('k', 'b').data

    left_defect_op_invariant = np.trace(
        np_fp
        @ np_left_defect_ops[0][1].conj().T
        @ np_fp
        @ np_left_defect_ops[0][1]
        @ np_left_rdm
    )

    right_defect_op_invariant = np.trace(
        np_fp
        @ np_right_defect_ops[0][1].conj().T
        @ np_fp
        @ np_right_defect_ops[0][1]
        @ np_right_rdm
    )

    np_defect_I = get_multisite_qu_I(
        domains_dict['num_defect_sites']
    ).transpose('k', 'b').data

    unitary_sym_op_list = [
        [np_defect_I, np_defect_I],
        [np_fp, np_fp]
    ]

    left_cocyles = get_cocycles_from_rho(
        np_left_rdm,
        np_left_defect_ops,
        unitary_sym_op_list
    )
    left_cocyles = gauge_fix_cocycles(left_cocyles)

    right_cocyles = get_cocycles_from_rho(
        np_right_rdm,
        np_right_defect_ops,
        unitary_sym_op_list
    )
    right_cocyles = gauge_fix_cocycles(right_cocyles)

    all_cocyles = np.array([left_cocyles, right_cocyles])

    cocycle_equation_output = compute_cocyle_equation_from_cocycles(all_cocyles)
    bosonic_cocycle_equation_output = (
        compute_bosonic_cocyle_equation_from_cocycles(all_cocyles)
    )

    #cocycle_invariants = (all_cocyles[:,0,0]**2)*(all_cocyles[:,1,1])

    fermion_decoration_from_cocycle = all_cocyles[:, 0, 1, 1, 0]/all_cocyles[:, 1, 0, 0, 1]
    """
    left_bosonic_cocycle = get_bosonic_cocycle(
        np_left_rdm,
        np_left_defect_ops[1],
        unitary_sym_op_list
    )
    right_bosonic_cocycle = get_bosonic_cocycle(
        np_right_rdm,
        np_right_defect_ops[1],
        unitary_sym_op_list
    )

    left_bosonic_cocyle_eq_from_boosnic_cocycle = (
        compute_bosonic_cocyle_equation_from_bosonic_cocycle(
            left_bosonic_cocycle
        )
    )
    right_bosonic_cocyle_eq_from_boosnic_cocycle = (
        compute_bosonic_cocyle_equation_from_bosonic_cocycle(
            right_bosonic_cocycle
        )
    )
    """

    out = {
        'left_proj_vec': left_proj_vec,
        'raw_left_proj_vec': raw_left_proj_vec,
        'left_proj_vec_results': left_proj_vec_results,
        'right_proj_vec': right_proj_vec,
        'raw_right_proj_vec': raw_right_proj_vec,
        'right_proj_vec_results': right_proj_vec_results,
        'cut_rho_trace': cut_rho_trace,
        'cut_rho_purity': cut_rho_purity,
        'sub_cut_rho_trace': sub_cut_rho_trace,
        'sub_cut_rho_purity': sub_cut_rho_purity,
        'cut_score': cut_score,
        'cut_state': cut_state,
        'cut_state_fermion_parity': cut_state_fermion_parity,
        'cut_overlap': cut_overlap,
        'defect_ops_scores': [l[1] for l in defect_ops_results],
        'left_defect_ops': left_defect_ops,
        'right_defect_ops': right_defect_ops,
        'left_defect_op_invariant': left_defect_op_invariant,
        'right_defect_op_invariant': right_defect_op_invariant,
        'cocycles': all_cocyles,
        'cocycle_equation_output': cocycle_equation_output,
        'bosonic_cocycle_equation_output': bosonic_cocycle_equation_output,
        #'cocycle_invariants': cocycle_invariants,
        'fermion_decoration_from_cocycle': fermion_decoration_from_cocycle
    }

    return out

## Find dominant eigenvector

In [64]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [65]:
def lanczos_iteration(rho, v, sites):
    w = (
        rho & v.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [66]:
def multiply_state_by_jw_string(state, sites):
    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in sites if (i%2 == 1)
    ]

    out = qtn.TensorNetwork(
        [
            *gates,
            state.reindex({f'k{i}': f'b{i}' for i in sites if (i%2 == 1)})
        ]
    )

    return out.contract()

In [67]:
def lanczos_iteration_jw_even(rho, v, sites):
    w = multiply_state_by_jw_string(v, sites)
    w = (v+w)/2

    w = (
        rho & w.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w = (multiply_state_by_jw_string(w, sites) + w)/2

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [68]:
def lanczos_algorithm(rho, v, sites, num_iters=20, jw_even=False):
    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    for _ in range(num_iters):
        v = update_func(rho, v, sites)

    return v

In [69]:
def get_dominant_eigenvector(rho, jw_even=False):
    k_inds = [i for i in rho.outer_inds() if i.startswith('k')]
    #b_inds = [i for i in rho.inds if i.startswith('b')]

    sites = [int(s[1:]) for s in k_inds]

    v = qtn.Tensor(
        data=random_uniform_complex((2,)*len(sites)),
        inds=[f'k{i}' for i in sites]
    )

    v = lanczos_algorithm(
        rho,
        v,
        sites,
        num_iters=20,
        jw_even=jw_even
    )

    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    new_v = update_func(rho, v, sites)

    overlap = np.abs((v & new_v.conj()).contract())

    score = (
        v.reindex({f'k{i}': f'b{i}' for i in sites})
        & rho
        & v.conj()
    )
    score = score.contract()


    return score, v, overlap

In [70]:
def compute_fermion_parity(state, sites):
    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in sites
    ]

    gate_exp = (
        state.reindex({f'k{i}': f'b{i}' for i in sites})
        & gates
        & state.conj()
    ).contract()

    return gate_exp

## Extract cocyles

In [71]:
def get_multisite_op(op, num_defect_sites):
    assert (num_defect_sites%2 == 0)

    np_op = multikron([op,]*(num_defect_sites//2))

    out = qtn.Tensor(
        np_op,
        inds=['k', 'b']
    )

    out = out.transpose('k', 'b').data

    return out

In [72]:
def normalize_defect_op(defect_op, defect_rho):
    exp = np.trace(
        defect_op
        @ defect_op
        @ defect_rho
    )

    phase = np.power(exp/np.abs(exp), -0.5)

    out = defect_op*phase

    return out

In [73]:
def get_cocycle(defect_rho, defect_op_list, unitary_sym_op_list, g_index,
                  h_index):
    g_fp_index, g_t_index = g_index
    h_fp_index, h_t_index = h_index
    
    g_op = defect_op_list[g_fp_index][g_t_index]

    h_op = defect_op_list[h_fp_index][h_t_index]
    h_op = h_op.conj() if (g_t_index%2 == 1) else h_op

    gh_fp_index = (g_fp_index+h_fp_index)%2
    gh_t_index = (g_t_index+h_t_index)%2
    gh_op = defect_op_list[gh_fp_index][gh_t_index]

    unitary_sym_op = unitary_sym_op_list[g_fp_index][g_t_index]

    out = np.trace(
        (gh_op.conj().T)
        @ unitary_sym_op
        @ h_op
        @ (unitary_sym_op.conj().T)
        @ g_op
        @ defect_rho
    )

    return out

In [74]:
def get_cocycles_from_rho(defect_rho, defect_op_list, unitary_sym_op_list):
    out = np.zeros((2,)*4, dtype=complex)

    for index in product([0,1], repeat=4):
        cocycle = get_cocycle(
            defect_rho,
            defect_op_list,
            unitary_sym_op_list,
            index[:2],
            index[2:]
        )

        out[*index] = cocycle

    return out

In [75]:
def gauge_fix_cocycles(cocycles):
    # Gauge fix cocycles to match appendix A.2 of Turzillo & You 2019
    L = np.array([
        [1, 1],
        [
            np.power(cocycles[1,0,1,0], 0.5),
            np.power(cocycles[1,0,1,0], 0.5)/cocycles[1,0,0,1]]
    ])

    gauge_transformation = np.ones((2,)*4, dtype=complex)

    for g_fp, g_t, h_fp, h_t in product([0,1], repeat=4):
        gh_fp = (g_fp + h_fp)%2
        gh_t = (g_t + h_t)%2
        num = L[gh_fp, gh_t]
        if (g_t%2 == 1):
            denom = (L[h_fp, h_t].conj())*L[g_fp, g_t]
        else:
            denom = (L[h_fp, h_t])*L[g_fp, g_t]
        gauge_transformation[g_fp, g_t, h_fp, h_t] = num/denom

    out = cocycles*gauge_transformation

    return out

In [76]:
def compute_cocyle_equation_from_cocycles(cocycles):
    #shape = cocyles
    # A bit janky, oh well
    #X = np.ones((2,)*5, dtype=complex)

    #X[:, 1:, 1:] = cocyles
    X=cocycles

    out = np.zeros((2,)*7, dtype=complex)

    for i_fp, i_t, j_fp, j_t, k_fp, k_t in product([0,1], repeat=6):
        num = X[...,i_fp, i_t, j_fp, j_t]*X[..., (i_fp+j_fp)%2, (i_t+j_t)%2, k_fp, k_t]
        if (i_t == 0):
            denom = X[...,j_fp, j_t, k_fp, k_t]*X[..., i_fp, i_t, (j_fp+k_fp)%2, (j_t+k_t)%2]
        if (i_t == 1):
            denom = (
                X[...,j_fp, j_t, k_fp, k_t].conj()
                *X[..., i_fp, i_t, (j_fp+k_fp)%2, (j_t+k_t)%2]
            )

        #print(num)
        #print(denom)

        out[:,i_fp, i_t, j_fp, j_t, k_fp, k_t] = num/denom

    return out

In [77]:
def compute_bosonic_cocyle_equation_from_cocycles(cocyles):
    #bosonic_cocycles = cocyles[..., :1, :1]
    #shape = bosonic_cocycles
    # A bit janky, oh well
    #X = np.ones((2, 2, 2), dtype=complex)

    #X[:, 1:, 1:] = bosonic_cocycles

    X = cocyles[:, 0, :, 0, :]

    out = np.zeros((2, 2, 2, 2), dtype=complex)

    for i,j,k in product(range(2), repeat=3):
        num = X[:,i,j]*X[:,(i+j)%2, k]
        if (i == 0):
            denom = X[:,j,k]*X[:, i, (j+k)%2]
        if (i == 1):
            denom = (X[:,j,k].conj())*X[:,i,(j+k)%2]

        out[:,i,j,k] = num/denom

    return out

# Test

In [78]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 4,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

## Product state

In [79]:
prod_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [80]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        prod_psi,
        domains_dict
    )
    results.append(current)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:17<00:00,  1.14it/s]


### Analyze results

#### Projector scores

In [81]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0.,  0., -0.,  0.,  0., -0.,  0., -0.,  0.,  0.,  0.,  0.,
       -0.,  0.,  0.,  0.,  0.,  0.,  0.])

In [82]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0., -0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,
       -0., -0.,  0.,  0.,  0.,  0.,  0.])

In [83]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [84]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.002, 0.   , 0.001, 0.   , 0.001, 0.001, 0.   , 0.001, 0.   ,
       0.001, 0.002, 0.   , 0.002, 0.   , 0.001, 0.002, 0.001, 0.003,
       0.001, 0.002])

In [85]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.001, 0.   , 0.   , 0.002, 0.002, 0.003, 0.004, 0.   , 0.002,
       0.002, 0.001, 0.001, 0.004, 0.   , 0.   , 0.001, 0.001, 0.001,
       0.002, 0.002])

#### Purities

In [86]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j, 0.999+0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j])

In [87]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

In [88]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j, 0.999+0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j])

Purities are the same, which one could likely prove.

In [89]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j, 0.999+0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j,
       1.   -0.j, 1.   +0.j])

#### Cut state results

In [90]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([-1.+1.38574976e-18j,  1.-1.94265092e-18j,  1.-1.90477263e-18j,
       -1.+1.81143340e-19j,  1.+4.96234108e-19j,  1.+4.50505720e-19j,
        1.+1.02350115e-18j, -1.-5.26416151e-18j, -1.-1.03998289e-18j,
       -1.-1.43319474e-18j, -1.-3.25903966e-19j, -1.-1.62775405e-18j,
        1.+6.83828892e-19j,  1.-1.59555536e-18j, -1.+2.55613056e-20j,
       -1.-7.16987228e-18j, -1.+2.41513702e-19j, -1.-1.65444394e-18j,
        1.-3.67824499e-18j,  1.-1.29509035e-18j])

So the FP of the cut state can be even or odd. Interesting!

In [91]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [92]:
np.round(np.array([
    [d['defect_ops_scores'][i][-1] for d in results]
    for i in range(3)
]), 5)

array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1.]])

#### Phases

In [93]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [94]:
left_phases.shape

(20,)

In [95]:
np.round(left_phases, 3)

array([1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j])

In [96]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [97]:
right_phases.shape

(20,)

In [98]:
np.round(right_phases, 3)

array([1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j])

#### Cocyle information

In [99]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
print(np.linalg.norm(cocycle_equation_array-1))
print(np.max(np.abs(cocycle_equation_array-1)))

0.0016784940500964762
0.0006662020144184469


In [100]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])

In [101]:
bosonic_cocycle_equation_array.shape

(20, 2, 2, 2, 2)

In [102]:
np.linalg.norm(bosonic_cocycle_equation_array-1)

np.float64(3.79836474257507e-05)

In [103]:
np.round(
    np.array([d['cocycles'][:, 0,1,0,1] for d in results])
, 3)

array([[1.-0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.-0.j]])

In [104]:
np.round(
    np.array([d['fermion_decoration_from_cocycle'] for d in results])
, 3)

array([[1.+0.j, 1.-0.j],
       [1.-0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.+0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.+0.j]])

## XZX state

In [105]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 10)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 4,
    'num_defect_sites': 4,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [106]:
xzx_psi = get_xzx_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [107]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        xzx_psi,
        domains_dict
    )
    results.append(current)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:37<00:00,  1.90s/it]


### Analyze results

#### Projector scores

In [108]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.,
       -0.,  0., -0., -0., -0., -0.,  0.])

In [109]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0., -0.,  0., -0., -0.,  0., -0., -0., -0., -0., -0., -0.,  0.,
       -0., -0., -0., -0., -0.,  0., -0.])

In [110]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [111]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.   , 0.007, 0.002, 0.001, 0.002, 0.001, 0.001, 0.001, 0.001,
       0.001, 0.003, 0.001, 0.001, 0.   , 0.002, 0.   , 0.001, 0.001,
       0.001, 0.002])

In [112]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.001, 0.   , 0.004, 0.001, 0.   , 0.008, 0.001, 0.   , 0.001,
       0.   , 0.   , 0.001, 0.006, 0.001, 0.002, 0.   , 0.001, 0.001,
       0.01 , 0.   ])

#### Purities

In [113]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [114]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [115]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [116]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

#### Cut state results

In [117]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([ 1.-2.58884082e-19j,  1.+6.34627749e-21j, -1.-2.69570164e-19j,
        1.-2.29399279e-20j,  1.-3.69952434e-20j,  1.-8.36861148e-21j,
        1.-2.38560367e-19j,  1.+5.73510330e-19j,  1.+6.70555208e-19j,
       -1.+4.51070163e-19j,  1.-2.59509515e-19j,  1.-1.42860037e-18j,
       -1.+1.28480578e-18j, -1.-2.63390747e-19j,  1.+1.67393289e-19j,
       -1.-2.28831161e-19j, -1.+1.68850106e-18j,  1.+1.64668602e-19j,
       -1.+1.15351221e-19j, -1.+1.64880379e-20j])

So the FP of the cut state can be even or odd. Interesting!

In [118]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [119]:
np.round(np.array([
    [d['defect_ops_scores'][i][-1] for d in results]
    for i in range(3)
]), 5)

array([[1.     , 1.     , 0.88242, 0.99901, 0.91517, 0.93927, 1.     ,
        1.     , 1.     , 1.     , 0.86848, 0.99804, 0.99875, 1.     ,
        0.99992, 0.99061, 1.     , 1.     , 1.     , 0.97841],
       [1.     , 1.     , 1.     , 1.     , 0.99974, 1.     , 1.     ,
        1.     , 1.     , 1.     , 1.     , 1.     , 0.9968 , 1.     ,
        1.     , 1.     , 1.     , 1.     , 1.     , 1.     ],
       [1.     , 1.     , 0.88242, 0.99901, 0.91742, 0.93927, 1.     ,
        1.     , 1.     , 1.     , 0.86848, 0.99804, 0.98667, 1.     ,
        0.99992, 0.99061, 1.     , 1.     , 1.     , 0.97841]])

#### Phases

In [120]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [121]:
left_phases.shape

(20,)

In [122]:
np.round(left_phases, 3)

array([-1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j, -0.907+0.j,
       -1.   +0.j, -0.052+0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j,
       -1.   +0.j, -1.   +0.j, -0.929-0.j, -1.   +0.j, -1.   +0.j,
       -1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j])

In [123]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [124]:
right_phases.shape

(20,)

In [125]:
np.round(right_phases, 3)

array([-1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j, -0.898-0.j,
       -1.   +0.j, -0.052-0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j,
       -1.   +0.j, -1.   +0.j, -0.924-0.j, -1.   +0.j, -1.   +0.j,
       -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j])

#### Cocyle information

In [126]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
print(np.linalg.norm(cocycle_equation_array-1))
print(np.max(np.abs(cocycle_equation_array-1)))

27.44916114031623
14.162475325327708


In [127]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])

In [128]:
bosonic_cocycle_equation_array.shape

(20, 2, 2, 2, 2)

In [129]:
np.linalg.norm(bosonic_cocycle_equation_array-1)

np.float64(1.8883652485496553e-05)

In [130]:
results[0]['cocycles'].shape

(2, 2, 2, 2, 2)

In [131]:
np.round(
    np.array([d['cocycles'][:, 0,1,0,1] for d in results])
, 3)

array([[1.-0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.-0.j, 1.-0.j],
       [1.-0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.-0.j],
       [1.+0.j, 1.+0.j],
       [1.-0.j, 1.-0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.+0.j],
       [1.-0.j, 1.+0.j]])

In [132]:
np.round(
    np.array([d['fermion_decoration_from_cocycle'] for d in results])
, 3)

array([[-1.   -0.j, -1.   +0.j],
       [-1.   +0.j, -1.   -0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.008+0.j, -1.009+0.j],
       [-1.   -0.j, -1.   +0.j],
       [ 0.37 +0.j,  0.37 +0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.   +0.j, -1.   -0.j],
       [-1.   +0.j, -1.   -0.j],
       [-0.83 +0.j, -0.82 +0.j],
       [-1.   +0.j, -1.   -0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   -0.j, -1.   +0.j]])

## YZY state

In [133]:
yzy_psi = get_yzy_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [134]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        yzy_psi,
        domains_dict
    )
    results.append(current)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:36<00:00,  1.80s/it]


### Analyze results

#### Projector scores

In [135]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0., -0., -0.,  0., -0., -0., -0., -0.,  0.,  0., -0.,  0., -0.,
       -0., -0.,  0.,  0.,  0., -0., -0.])

In [136]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0., -0., -0.,  0., -0., -0., -0.,  0., -0., -0., -0., -0., -0.,
        0., -0.,  0.,  0., -0., -0., -0.])

In [137]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [138]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.001, 0.001, 0.001, 0.004, 0.   , 0.   , 0.   , 0.   , 0.004,
       0.001, 0.003, 0.001, 0.001, 0.001, 0.001, 0.002, 0.001, 0.   ,
       0.002, 0.001])

In [139]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.001, 0.001, 0.   , 0.003, 0.003, 0.001, 0.003, 0.003, 0.002,
       0.   , 0.001, 0.002, 0.   , 0.009, 0.002, 0.005, 0.005, 0.   ,
       0.   , 0.   ])

#### Purities

In [140]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j])

In [141]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [142]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [143]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j])

#### Cut state results

In [144]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([-1.+4.29393329e-19j, -1.+2.86374872e-19j,  1.-1.07699859e-18j,
       -1.-6.12766559e-19j,  1.-4.91525280e-19j,  1.+2.37488532e-19j,
       -1.+5.43076554e-20j, -1.-2.16729044e-19j,  1.+1.25645009e-18j,
       -1.+6.11201967e-19j, -1.+1.12279115e-19j,  1.+1.62771896e-19j,
       -1.-5.72644278e-19j, -1.-3.59593311e-19j, -1.+6.73629076e-19j,
        1.+1.76787147e-19j, -1.+1.65268338e-19j,  1.+9.88520774e-20j,
        1.+8.22055130e-19j, -1.-4.98459483e-19j])

So the FP of the cut state can be even or odd. Interesting!

In [145]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [146]:
np.round(np.array([
    [d['defect_ops_scores'][i][-1] for d in results]
    for i in range(3)
]), 5)

array([[0.99903, 0.95215, 1.     , 1.     , 0.95036, 1.     , 1.     ,
        1.     , 1.     , 0.94981, 1.     , 0.88243, 1.     , 0.99614,
        1.     , 0.9964 , 0.8243 , 0.98328, 1.     , 1.     ],
       [0.99986, 0.99998, 1.     , 1.     , 1.     , 1.     , 1.     ,
        1.     , 1.     , 1.     , 1.     , 1.     , 1.     , 1.     ,
        1.     , 0.98   , 1.     , 1.     , 1.     , 1.     ],
       [0.99906, 0.9709 , 1.     , 1.     , 0.94867, 1.     , 1.     ,
        1.     , 1.     , 0.94981, 1.     , 0.88435, 1.     , 0.99614,
        1.     , 0.96864, 0.8243 , 0.98328, 1.     , 1.     ]])

#### Phases

In [147]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [148]:
left_phases.shape

(20,)

In [149]:
np.round(left_phases, 3)

array([ 0.997-0.j, -0.487-0.j, -1.   +0.j, -1.   +0.j,  0.999+0.j,
       -1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j,
       -1.   -0.j,  0.554-0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j,
       -0.78 +0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j])

In [150]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [151]:
right_phases.shape

(20,)

In [152]:
np.round(right_phases, 3)

array([ 0.997-0.j, -0.431+0.j, -1.   +0.j, -1.   -0.j,  0.999+0.j,
       -1.   -0.j, -1.   +0.j, -1.   +0.j, -1.   -0.j, -1.   -0.j,
       -1.   +0.j,  0.557+0.j, -1.   -0.j, -1.   +0.j, -1.   +0.j,
       -0.768+0.j, -1.   +0.j, -1.   +0.j, -1.   -0.j, -1.   +0.j])

#### Cocyle information

In [153]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
print(np.linalg.norm(cocycle_equation_array-1))
print(np.max(np.abs(cocycle_equation_array-1)))

466.4968381771777
213.6953539293931


In [154]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])

In [155]:
bosonic_cocycle_equation_array.shape

(20, 2, 2, 2, 2)

In [156]:
np.linalg.norm(bosonic_cocycle_equation_array-1)

np.float64(1.6981513252458836e-05)

In [157]:
results[0]['cocycles'].shape

(2, 2, 2, 2, 2)

In [158]:
np.round(
    np.array([d['cocycles'][:, 0,1,0,1] for d in results])
, 3)

array([[ 0.997+0.j,  0.997+0.j],
       [-0.487-0.j, -0.431-0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.   -0.j, -1.   -0.j],
       [ 0.999+0.j,  0.999-0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.   +0.j, -1.   -0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   +0.j, -1.   +0.j],
       [ 0.554-0.j,  0.557-0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   +0.j, -1.   -0.j],
       [-0.78 +0.j, -0.768+0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   +0.j, -1.   -0.j],
       [-1.   +0.j, -1.   +0.j]])

In [159]:
np.round(
    np.array([d['fermion_decoration_from_cocycle'] for d in results])
, 3)

array([[ 1.054+0.j,  1.054-0.j],
       [-0.986+0.j, -0.984-0.j],
       [-1.   +0.j, -1.   -0.j],
       [-1.   +0.j, -1.   +0.j],
       [ 1.011-0.j,  1.012+0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   +0.j, -1.   -0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   -0.j, -1.   +0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.   +0.j, -1.   -0.j],
       [12.724+0.j, 12.201+0.j],
       [-1.   +0.j, -1.   +0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   -0.j, -1.   +0.j],
       [-0.613-0.j, -0.612+0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   -0.j, -1.   -0.j],
       [-1.   +0.j, -1.   -0.j],
       [-1.   +0.j, -1.   -0.j]])

## XZZZX state

In [223]:
domains_dict = {
    'num_system_sites': 40,
    'left_projector_sites': list(range(4, 12)),
    'right_projector_sites': list(range(28, 36)),
    'num_projector_pad_sites': 4,
    'num_defect_sites': 5,
    'fdlu_depth': 2,
    'fdlu_offset': 0,
}

In [224]:
xzzzx_psi = get_xzzzx_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [225]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        xzzzx_psi,
        domains_dict
    )
    results.append(current)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [02:36<00:00,  7.82s/it]


### Analyze results

#### Projector scores

In [226]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0., -0., -0., -0., -0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
       -0.,  0.,  0.,  0.,  0., -0., -0.])

In [227]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0., -0.,  0., -0., -0., -0.,  0.,  0., -0., -0., -0., -0.,  0.,
       -0.,  0.,  0., -0.,  0.,  0.,  0.])

In [228]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [229]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.001, 0.003, 0.001, 0.003, 0.   , 0.001, 0.005, 0.003, 0.004,
       0.003, 0.001, 0.003, 0.002, 0.002, 0.004, 0.004, 0.015, 0.004,
       0.004, 0.001])

In [230]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.004, 0.001, 0.001, 0.003, 0.001, 0.001, 0.004, 0.004, 0.002,
       0.001, 0.002, 0.001, 0.004, 0.001, 0.002, 0.001, 0.002, 0.003,
       0.004, 0.003])

#### Purities

In [231]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.   -0.j, 0.517+0.j, 0.801-0.j, 1.   +0.j, 0.734-0.j, 0.783-0.j,
       0.985-0.j, 0.953+0.j, 0.972-0.j, 0.516-0.j, 0.362-0.j, 0.551-0.j,
       0.667+0.j, 0.733-0.j, 0.785+0.j, 0.536+0.j, 0.47 +0.j, 0.507+0.j,
       0.558-0.j, 0.994+0.j])

In [232]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [233]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.   -0.j, 0.517+0.j, 0.801-0.j, 1.   +0.j, 0.734-0.j, 0.783-0.j,
       0.985-0.j, 0.953+0.j, 0.972-0.j, 0.516-0.j, 0.362-0.j, 0.551-0.j,
       0.667+0.j, 0.733-0.j, 0.785+0.j, 0.536+0.j, 0.47 +0.j, 0.507+0.j,
       0.558-0.j, 0.994+0.j])

Purities are the same, which one could likely prove.

In [234]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.   -0.j, 0.517+0.j, 0.801-0.j, 1.   +0.j, 0.734-0.j, 0.783-0.j,
       0.985-0.j, 0.953+0.j, 0.972-0.j, 0.516-0.j, 0.362-0.j, 0.551-0.j,
       0.667+0.j, 0.733-0.j, 0.785+0.j, 0.536+0.j, 0.47 +0.j, 0.507+0.j,
       0.558-0.j, 0.994+0.j])

#### Cut state results

In [235]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([-1.        -1.12355074e-19j,  0.99999922+3.15777634e-19j,
       -1.        +2.05547739e-19j, -1.        -2.05948651e-19j,
       -1.        +3.20946016e-20j,  1.        -2.62429554e-19j,
        1.        +2.14304787e-19j, -1.        -6.94009179e-20j,
       -1.        +5.62671792e-20j,  0.99999385-1.07502340e-19j,
        0.99990958+1.38323653e-21j,  1.        -8.11780684e-20j,
       -1.        +3.59193629e-20j, -1.        +1.97643881e-19j,
       -1.        -6.70471282e-20j, -1.        +5.99408809e-20j,
        0.99999994+1.61793013e-19j, -1.        +2.86121441e-21j,
        1.        -2.14030038e-19j, -1.        +4.14635542e-20j])

In [236]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [237]:
np.round(np.array([
    [d['defect_ops_scores'][i][-1] for d in results]
    for i in range(3)
]), 5)

array([[0.99924, 1.     , 0.99699, 1.     , 1.     , 1.     , 1.     ,
        1.     , 1.     , 0.95554, 0.87154, 0.96612, 1.     , 1.     ,
        0.92313, 0.77935, 1.     , 0.99974, 0.01758, 1.     ],
       [1.     , 1.     , 1.     , 1.     , 1.     , 1.     , 1.     ,
        1.     , 1.     , 1.     , 0.99999, 1.     , 1.     , 1.     ,
        1.     , 1.     , 1.     , 1.     , 1.     , 1.     ],
       [0.99924, 1.     , 0.99699, 1.     , 1.     , 1.     , 1.     ,
        1.     , 1.     , 0.95554, 0.87155, 0.96612, 1.     , 1.     ,
        0.92313, 0.77935, 1.     , 0.99974, 0.01758, 1.     ]])

#### Phases

In [238]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [239]:
left_phases.shape

(20,)

In [240]:
np.round(left_phases, 3)

array([1.   -0.j, 0.769+0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       0.995-0.j, 0.995+0.j, 1.   -0.j, 0.775-0.j, 1.   -0.j, 1.   -0.j,
       1.   +0.j, 1.   -0.j, 1.   +0.j, 0.999-0.j, 0.912-0.j, 0.785-0.j,
       1.   -0.j, 1.   +0.j])

In [241]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [242]:
right_phases.shape

(20,)

In [243]:
np.round(right_phases, 3)

array([1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j, 0.936-0.j,
       1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 0.975-0.j, 0.999-0.j,
       0.927+0.j, 0.904-0.j, 0.95 +0.j, 0.855-0.j, 1.   -0.j, 0.988+0.j,
       1.   -0.j, 0.999+0.j])

#### Cocyle information

In [244]:
cocycle_equation_array = np.array([d['cocycle_equation_output'] for d in results])
print(np.linalg.norm(cocycle_equation_array-1))
print(np.max(np.abs(cocycle_equation_array-1)))

3.708694691304267
0.8754966668865869


In [245]:
bosonic_cocycle_equation_array = np.array([d['bosonic_cocycle_equation_output'] for d in results])

In [246]:
bosonic_cocycle_equation_array.shape

(20, 2, 2, 2, 2)

In [247]:
np.linalg.norm(bosonic_cocycle_equation_array-1)

np.float64(0.15280640705382523)

In [248]:
results[0]['cocycles'].shape

(2, 2, 2, 2, 2)

In [249]:
np.round(
    np.array([d['cocycles'][:, 0,1,0,1] for d in results])
, 3)

array([[-1.   +0.j   , -1.   +0.j   ],
       [-0.747-0.04j , -1.   +0.j   ],
       [-1.   +0.j   , -0.999-0.j   ],
       [-1.   +0.j   , -1.   -0.j   ],
       [-1.   -0.j   , -1.   -0.j   ],
       [-1.   +0.j   , -0.905-0.001j],
       [-0.994+0.j   , -1.   -0.j   ],
       [-0.994-0.j   , -1.   +0.j   ],
       [-1.   -0.j   , -1.   -0.j   ],
       [-0.721-0.012j, -1.   -0.j   ],
       [-1.   -0.j   , -0.956+0.001j],
       [-1.   -0.j   , -0.999+0.j   ],
       [-1.   -0.j   , -0.881+0.001j],
       [-1.   +0.j   , -0.883-0.003j],
       [-1.   +0.j   , -0.939+0.001j],
       [-0.999-0.j   , -0.798+0.014j],
       [-0.877-0.008j, -1.   +0.j   ],
       [-0.687+0.033j, -0.98 -0.003j],
       [-1.   +0.j   , -0.997+0.j   ],
       [-1.   -0.j   , -0.998+0.j   ]])

In [250]:
np.round(
    np.array([d['fermion_decoration_from_cocycle'] for d in results])
, 3)

array([[1.   +0.j   , 1.   -0.j   ],
       [1.019-0.05j , 1.   +0.j   ],
       [1.   -0.j   , 1.   -0.j   ],
       [1.   -0.j   , 1.   +0.j   ],
       [1.   -0.j   , 1.   +0.j   ],
       [1.   -0.j   , 1.   +0.016j],
       [0.999+0.j   , 1.   -0.j   ],
       [1.   -0.001j, 1.   -0.j   ],
       [1.   +0.j   , 1.   -0.j   ],
       [0.964+0.06j , 1.   -0.j   ],
       [1.   +0.j   , 1.002+0.002j],
       [1.   -0.j   , 1.   +0.j   ],
       [1.   -0.j   , 1.001+0.019j],
       [1.   -0.j   , 1.046+0.01j ],
       [1.   -0.j   , 1.002-0.006j],
       [1.   +0.j   , 1.011-0.062j],
       [1.009-0.025j, 1.   -0.j   ],
       [0.99 -0.041j, 1.001-0.001j],
       [1.   +0.j   , 1.   -0.j   ],
       [1.   -0.j   , 1.   +0.j   ]])

# Conclusions
Looks good!